# Step 4 - Moment Retrieval Using Pre-trained Models

- Video 5 is about 111 seconds, so it can be encoded directly by Lighthouse.
- Video 6 is about 323 seconds, while Lighthouse's benchmark-style raw-video path is safest below about 150 seconds, so the notebook chunks video 6 and maps local chunk predictions back to global timestamps.

## 1. Setup

In [71]:
from pathlib import Path
import json
import math
import os
import re
import shutil
import sys
import warnings
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")
import cv2
import ffmpeg
import ffmpeg.nodes as ffmpeg_nodes
import imageio_ffmpeg
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Video, display
try:
    from moviepy import VideoFileClip
except ImportError:
    from moviepy.editor import VideoFileClip

# The trusted Lighthouse checkpoints include pickled option objects, so they
# need PyTorch's pre-2.6 checkpoint loading behavior.

# The ffmpeg-python wrapper normally shells out to ffmpeg/ffprobe. In this
# Windows conda setup, imageio-ffmpeg provides a working ffmpeg binary, and
# OpenCV is reliable for the small metadata block Lighthouse needs.
FFMPEG_EXE = imageio_ffmpeg.get_ffmpeg_exe()
if not hasattr(ffmpeg, "_codex_original_run"):
    ffmpeg._codex_original_run = ffmpeg.run
    ffmpeg._codex_original_run_async = ffmpeg.run_async

def _run_with_imageio_ffmpeg(stream_spec, cmd=FFMPEG_EXE, **kwargs):
    return ffmpeg._codex_original_run(stream_spec, cmd=cmd, **kwargs)

def _run_async_with_imageio_ffmpeg(stream_spec, cmd=FFMPEG_EXE, **kwargs):
    return ffmpeg._codex_original_run_async(stream_spec, cmd=cmd, **kwargs)

ffmpeg.run = _run_with_imageio_ffmpeg
ffmpeg.run_async = _run_async_with_imageio_ffmpeg
ffmpeg_nodes.OutputStream.run = _run_with_imageio_ffmpeg
ffmpeg_nodes.OutputStream.run_async = _run_async_with_imageio_ffmpeg

from lighthouse.frame_loaders.base_loader import BaseLoader

def _video_info_with_opencv(self, video_path):
    capture = cv2.VideoCapture(str(video_path))
    try:
        if not capture.isOpened():
            return None

        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps_value = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
        frames_length = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

        if width <= 0 or height <= 0 or fps_value <= 0 or frames_length <= 0:
            return None

        return {
            "duration": frames_length / fps_value,
            "frames_length": frames_length,
            "fps": max(1, math.floor(fps_value)),
            "height": height,
            "width": width,
        }
    finally:
        capture.release()

BaseLoader._video_info = _video_info_with_opencv

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="CUDA initialization:.*")

#display all rows and columns in pandas DataFrames
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

ROOT = Path.cwd()
VIDEO_DIR = ROOT / "TVSum"
ANNOTATION_DIR = ROOT / "annotations"
PREDICTION_SOURCE = ROOT / "video6_step3_predictions.txt"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
VIDEO_IDS_TO_RUN = [6]  
DEVICE = "cpu"

In [72]:
from lighthouse.models import CGDETRPredictor, QDDETRPredictor, MomentDETRPredictor

MODEL_CONFIGS = {
    "CG-DETR": {
        "predictor_cls": CGDETRPredictor,
        "checkpoint": ROOT / "clip_cg_detr_qvhighlight.ckpt",
        "feature_name": "clip",
        "slowfast_path": None,
        "selected": True,
    },
    "QD-DETR": {
        "predictor_cls": QDDETRPredictor,
        "checkpoint": ROOT / "qd_detr_qvhighlight_videoonly.ckpt",
        # This checkpoint expects 2818-dim video input:
        # 512 CLIP + 2304 SlowFast + 2 temporal endpoint features.
        "feature_name": "clip_slowfast",
        "slowfast_path": ROOT / "SLOWFAST_8x8_R50.pkl",
        "selected": True,
    },
    "Moment-DETR": {
        "predictor_cls": MomentDETRPredictor,
        "checkpoint": ROOT / "moment_detr_qvhighlight_clip.ckpt",
        "feature_name": "clip",
        "slowfast_path": None,
        "selected": True,
    },
}

## 2. Parse Videos and Annotations

The annotation files are parsed into event text, start/end seconds, and subjectivity score. The default query table keeps events with score `>= 1`, because those are the objectively important/essential moments most relevant to a summary.

In [73]:
def mmss_to_seconds(value):

    value = value.strip()

    minutes, seconds = value.split(":")

    return int(minutes) * 60 + int(seconds)


def seconds_to_mmss(seconds):

    seconds = max(0, int(round(seconds)))

    return f"{seconds // 60:02d}:{seconds % 60:02d}"

TIME_RE = re.compile(r"(\d{1,2}:\d{2})\s*(?:-|\u2013|\u2014|\?|\uFFFD)+\s*(\d{1,2}:\d{2})")

def infer_video_id(path):

    name = path.name.lower()

    if "video_5" in name or "video5" in name:

        return 5

    if re.search(r"(^|\D)5(\D|$)", name):

        return 5
    
    if "video_6" in name or "video6" in name:

        return 6

    if re.search(r"(^|\D)6(\D|$)", name):

        return 6
    
    return None


def parse_annotation_line(line, source_file, row_index):

    cleaned = line.strip().replace("\u2013", "-").replace("\u2014", "-").replace("\ufffd", "-")

    if not cleaned:

        return None

    time_match = TIME_RE.search(cleaned)

    score_match = re.search(r",\s*(-?\d+)\s*$", cleaned)

    if not time_match or not score_match:

        return None

    event = cleaned[: time_match.start()].strip(" ,")

    start_s = mmss_to_seconds(time_match.group(1))

    end_s = mmss_to_seconds(time_match.group(2))

    score = int(score_match.group(1))

    video_id = infer_video_id(source_file)


    return {

        "video_id": video_id,
        "source_file": source_file.name,
        "row_index": row_index,
        "event": event,
        "gt_start": start_s,
        "gt_end": end_s,
        "gt_time": f"{seconds_to_mmss(start_s)} - {seconds_to_mmss(end_s)}",
        "subjectivity": score,

    }


def parse_prediction_line(line, source_file, row_index):

    cleaned = line.strip().replace("\u2013", "-").replace("\u2014", "-").replace("\ufffd", "-")

    if not cleaned:

        return None

    time_match = TIME_RE.search(cleaned)

    if not time_match:

        return None

    event = cleaned[: time_match.start()].strip(" ,")

    event = re.sub(r"^Predicted event\s+\d+\s*:\s*", "", event, flags=re.IGNORECASE).strip()

    start_s = mmss_to_seconds(time_match.group(1))

    end_s = mmss_to_seconds(time_match.group(2))

    video_id = infer_video_id(source_file)

    return {

        "video_id": video_id,
        "source_file": source_file.name,
        "row_index": row_index,
        "event": event,
        "gt_start": start_s,
        "gt_end": end_s,
        "gt_time": f"{seconds_to_mmss(start_s)} - {seconds_to_mmss(end_s)}",
        "subjectivity": 1,

    }


def iter_text_files(path_or_dir):

    path_or_dir = Path(path_or_dir)

    if path_or_dir.is_file():

        return [path_or_dir]

    if path_or_dir.is_dir():

        return sorted(path_or_dir.glob("*.txt"))

    return []


def load_predictions(prediction_source):

    rows = []

    for path in iter_text_files(prediction_source):

        video_id = infer_video_id(path)

        if video_id not in {5, 6}:

            continue

        for row_index, line in enumerate(path.read_text(encoding="utf-8", errors="replace").splitlines(), 1):

            parsed = parse_prediction_line(line, path, row_index)

            if parsed:

                rows.append(parsed)

    return pd.DataFrame(rows, columns=["video_id", "source_file", "row_index", "event", "gt_start", "gt_end", "gt_time", "subjectivity"])


def load_annotations(annotation_dir):

    rows = []

    for path in sorted(annotation_dir.glob("*.txt")):

        video_id = infer_video_id(path)

        if video_id not in {5, 6}:

            continue

        for row_index, line in enumerate(path.read_text(encoding="utf-8", errors="replace").splitlines(), 1):

            parsed = parse_annotation_line(line, path, row_index)

            if parsed:

                rows.append(parsed)

    return pd.DataFrame(rows)


annotations_df = load_annotations(ANNOTATION_DIR)
predictions_df = load_predictions(PREDICTION_SOURCE)


predictions_df if not predictions_df.empty else annotations_df

,video_id,source_file,row_index,event,gt_start,gt_end,gt_time,subjectivity
0,6,video6_step3_predictions.txt,1,The video begins with a view of a wooden patio...,0,10,00:00 - 00:10,1
1,6,video6_step3_predictions.txt,2,"The camera follows the truck as it moves, capt...",31,42,00:31 - 00:42,1
2,6,video6_step3_predictions.txt,3,The RV is secured with a yellow rope tied to a...,43,52,00:43 - 00:52,1
3,6,video6_step3_predictions.txt,4,"The truck is tilted to the side, and the camer...",86,88,01:26 - 01:28,1
4,6,video6_step3_predictions.txt,5,The second truck reversing close to the stuck ...,152,168,02:32 - 02:48,1
5,6,video6_step3_predictions.txt,6,The truck is white with a blue stripe and has ...,224,266,03:44 - 04:26,1
6,6,video6_step3_predictions.txt,7,The truck continues to move down the road,224,266,03:44 - 04:26,1


In [74]:
import glob

def split_video_into_chunks(video_path, output_dir, num_chunks=10):

    video_path = str(video_path)
    output_dir = str(output_dir)

    os.makedirs(output_dir, exist_ok=True)

    # Reuse existing chunks if available
    existing_chunks = sorted(glob.glob(os.path.join(output_dir, "chunk_*.mp4")))

    if existing_chunks:

        chunk_paths = []

        for chunk_path in existing_chunks:

            filename = os.path.basename(chunk_path)

            parts = (
                filename
                .replace("chunk_", "")
                .replace(".mp4", "")
                .split("_")
            )

            start = float(parts[0])
            end = float(parts[1])

            chunk_paths.append((chunk_path, start, end))

        print(f"Reusing {len(chunk_paths)} existing chunks from {output_dir}")

        return chunk_paths

    # Load video
    video = VideoFileClip(video_path)

    duration = video.duration

    # Compute equal chunk duration
    chunk_duration = duration / num_chunks

    chunk_paths = []

    for i in range(num_chunks):

        start = i * chunk_duration

        # ensure final chunk reaches exact end
        if i == num_chunks - 1:
            end = duration
        else:
            end = (i + 1) * chunk_duration



        chunk_path = os.path.join(
            output_dir,
        f"chunk_{start:07.2f}_{end:07.2f}.mp4"
        )


        # compatibility with moviepy versions
        if hasattr(video, "subclipped"):
            clip = video.subclipped(start, end)
        else:
            clip = video.subclip(start, end)

        clip.write_videofile(
            chunk_path,
            codec="libx264",
            audio=False,
            logger=None
        )

        clip.close()

        chunk_paths.append((chunk_path, start, end))

    video.close()

    return chunk_paths


def get_video_duration(video_path) :

    clip = VideoFileClip(str(video_path))

    try:

        return float(clip.duration)
    
    finally:

        clip.close()


video_info = []

for video_id in VIDEO_IDS_TO_RUN:

    path = VIDEO_DIR / f"video_{video_id}.mp4"

    duration = get_video_duration(path)

    video_info.append(

        {
            "video_id": video_id,
            "path": str(path.relative_to(ROOT)),
            "duration_seconds": duration,
            "duration_mmss": seconds_to_mmss(duration),
        }
    )

num_chunks = 3

output_dir = f"video_{VIDEO_IDS_TO_RUN[0]}_{num_chunks}_chunks"

chunks = split_video_into_chunks(
    VIDEO_DIR / f"video_{video_id}.mp4",
    output_dir=output_dir,
    num_chunks= num_chunks# changed to 8 seconds
)


video_info_df = pd.DataFrame(video_info)
video_info_df

Reusing 3 existing chunks from video_6_3_chunks


,video_id,path,duration_seconds,duration_mmss
0,6,TVSum\video_6.mp4,322.71,05:23


In [75]:
# Use Step 3 predicted events when available; otherwise fall back to manual annotations.
query_source_df = predictions_df if not predictions_df.empty else annotations_df
query_source_name = "Step 3 predictions" if not predictions_df.empty else "manual annotations"

query_table = (
    query_source_df[
        (query_source_df["video_id"].isin(VIDEO_IDS_TO_RUN))
        & (query_source_df["subjectivity"] >= 1)
    ]
    .sort_values(["video_id", "gt_start", "gt_end", "source_file"])
    .reset_index(drop=True)
)

print(f"Using {query_source_name}")
print(query_table.groupby("video_id").size())
query_table[["video_id", "event", "gt_time", "subjectivity", "source_file"]]

Using Step 3 predictions
video_id
6    7
dtype: int64


,video_id,event,gt_time,subjectivity,source_file
0,6,The video begins with a view of a wooden patio...,00:00 - 00:10,1,video6_step3_predictions.txt
1,6,"The camera follows the truck as it moves, capt...",00:31 - 00:42,1,video6_step3_predictions.txt
2,6,The RV is secured with a yellow rope tied to a...,00:43 - 00:52,1,video6_step3_predictions.txt
3,6,"The truck is tilted to the side, and the camer...",01:26 - 01:28,1,video6_step3_predictions.txt
4,6,The second truck reversing close to the stuck ...,02:32 - 02:48,1,video6_step3_predictions.txt
5,6,The truck is white with a blue stripe and has ...,03:44 - 04:26,1,video6_step3_predictions.txt
6,6,The truck continues to move down the road,03:44 - 04:26,1,video6_step3_predictions.txt


## 3. Retrieval Utilities

Video 5 is short enough for direct retrieval. Video 6 is automatically split into chunks and each chunk's predictions are shifted back to the original video timeline.

In [76]:
MAX_DIRECT_VIDEO_SECONDS = 150
CHUNK_SECONDS = 30
TOP_K_PER_QUERY = 5
NMS_IOU_THRESHOLD = 0.70

QUERY_VARIANT_STRATEGIES_TO_RUN = [
    "raw_vlm",
    "cleaned",
    "short_action",
    "the_video_shows",
    "find_moment",
    "screen_text",
]

QUERY_VARIANT_DESCRIPTIONS = {
    "raw_vlm": "Original Step 3 event sentence.",
    "cleaned": "Remove VLM narration such as 'the video begins with'.",
    "short_action": "Keep the most direct subject/action/object clause.",
    "the_video_shows": "Wrap the cleaned event as 'The video shows ...'.",
    "find_moment": "Wrap the cleaned event as 'Find the moment showing ...'.",
    "screen_text": "Emphasize quoted text or on-screen text when present.",
}


def temporal_iou(a_start, a_end, b_start, b_end):

    inter = max(0.0, min(a_end, b_end) - max(a_start, b_start))

    union = max(a_end, b_end) - min(a_start, b_start)
    
    return inter / union if union > 0 else 0.0


def nms_windows(windows, iou_threshold=0.70, max_keep=5):

    ordered = sorted(windows, key=lambda item: item["score"], reverse=True)

    kept = []

    for item in ordered:

        overlaps = [
            temporal_iou(item["pred_start"], item["pred_end"], prev["pred_start"], prev["pred_end"])
            for prev in kept
        ]

        if not overlaps or max(overlaps) < iou_threshold:

            kept.append(item)

        if len(kept) >= max_keep:

            break

    return kept


def list_existing_chunks(chunk_dir):

    chunk_dir = Path(chunk_dir)

    if not chunk_dir.exists():

        return []
    
    chunks = []

    for path in sorted(chunk_dir.glob("chunk_*.mp4")):

        match = re.match(r"chunk_(\d+(?:\.\d+)?)_(\d+(?:\.\d+)?)\.mp4$", path.name)

        if not match:

            continue

        start = float(match.group(1))
        end = float(match.group(2))

        if end <= start:

            continue

        chunks.append({"path": path, "offset": start, "end": end})

    return sorted(chunks, key=lambda item: (item["offset"], item["end"]))


def chunks_cover_video(chunks, video_duration, max_gap_seconds=1.5):

    if not chunks or video_duration <= 0:

        return False

    cursor = 0.0

    for chunk in sorted(chunks, key=lambda item: item["offset"]):

        start = max(0.0, min(float(video_duration), float(chunk["offset"])))
        end = max(0.0, min(float(video_duration), float(chunk["end"])))

        if end <= start:

            continue

        if start > cursor + max_gap_seconds:

            return False

        cursor = max(cursor, end)

    return cursor >= float(video_duration) - max_gap_seconds


def find_existing_chunk_sets(video_id, video_duration):

    candidate_dirs = [ROOT / f"video_{video_id}_chunks"]

    for pattern in [
        f"video_{video_id}_chunks*",
        f"video_{video_id}_*_chunks",
        f"video_{video_id}_*_chunks*",
    ]:

        candidate_dirs.extend(sorted(path for path in ROOT.glob(pattern) if path.is_dir()))

    seen = set()
    chunk_sets = []

    for chunk_dir in candidate_dirs:

        chunk_dir = Path(chunk_dir)
        resolved = chunk_dir.resolve()

        if resolved in seen:

            continue

        seen.add(resolved)

        chunks = list_existing_chunks(chunk_dir)

        if not chunks:

            continue

        chunk_sets.append(
            {
                "chunk_dir": chunk_dir,
                "chunks": chunks,
                "complete": chunks_cover_video(chunks, video_duration),
                "last_end": max(chunk["end"] for chunk in chunks),
            }
        )

    return chunk_sets


def choose_existing_chunks(video_id, video_duration):

    chunk_sets = find_existing_chunk_sets(video_id, video_duration)

    complete_sets = [chunk_set for chunk_set in chunk_sets if chunk_set["complete"]]

    if not complete_sets:

        return []

    # Prefer the most granular complete chunk set, since retrieval scores are local to each segment.
    selected = max(complete_sets, key=lambda item: (len(item["chunks"]), item["last_end"]))
    print(f"Using existing chunks from {selected['chunk_dir'].relative_to(ROOT)}")

    return selected["chunks"]


def split_video_into_chunks(video_path, output_dir, chunk_seconds=30):

    video_path = Path(video_path)
    output_dir = Path(output_dir)

    output_dir.mkdir(exist_ok=True)

    clip = VideoFileClip(str(video_path))

    chunks = []

    try:

        duration = int(np.ceil(clip.duration))

        for start in range(0, duration, chunk_seconds):

            end = min(start + chunk_seconds, duration)

            chunk_path = output_dir / f"chunk_{start:04d}_{end:04d}.mp4"

            if not chunk_path.exists():

                subclip = clip.subclipped(start, end) if hasattr(clip, "subclipped") else clip.subclip(start, end)

                try:

                    subclip.write_videofile(

                        str(chunk_path),
                        codec="libx264",
                        audio=False,
                        logger=None,
                    )

                finally:

                    subclip.close()

            chunks.append({"path": chunk_path, "offset": float(start), "end": float(end)})

    finally:

        clip.close()

    return chunks


def get_video_segments(video_id: int):

    video_path = VIDEO_DIR / f"video_{video_id}.mp4"

    duration = get_video_duration(video_path)

    existing_chunks = choose_existing_chunks(video_id, duration)

    if existing_chunks:

        return existing_chunks, duration

    if duration <= MAX_DIRECT_VIDEO_SECONDS:

        return [{"path": video_path, "offset": 0.0, "end": duration}], duration
    

    chunk_dir = ROOT / f"video_{video_id}_chunks"

    chunks = list_existing_chunks(chunk_dir)

    if not chunks:

        chunks = split_video_into_chunks(video_path, chunk_dir, CHUNK_SECONDS)

    return chunks, duration

def normalize_query_text(text):

    return re.sub(r"\s+", " ", str(text)).strip(" .,")


def lower_first(text):

    text = normalize_query_text(text)

    return text[:1].lower() + text[1:] if text else text


def clean_vlm_event(event):

    text = normalize_query_text(event)

    prefix_re = re.compile(
        r"^(the video begins with|the video starts with|the video continues with|"
        r"the final segment shows|the video concludes with|it transitions to|"
        r"the segment shows|this segment shows)\s+",
        flags=re.IGNORECASE,
    )

    previous = None

    while previous != text:

        previous = text

        text = prefix_re.sub("", text).strip(" .,;")

    return text or normalize_query_text(event)


def short_action_query(event):

    text = clean_vlm_event(event)

    # Step 3 descriptions often contain multiple clauses; the first visual clause is usually the cleanest query.
    text = re.split(r"\s*(?:, followed by|, with|\. It |;|,)\s*", text, maxsplit=1)[0]

    return normalize_query_text(text)


def screen_text_query(event):

    text = clean_vlm_event(event)

    quoted = re.findall(r'"([^"]+)"', text)

    if quoted:

        return normalize_query_text("screen text " + " ".join(quoted))

    if re.search(r"\b(text|title|screen|website|logo|display|appears?)\b", text, flags=re.IGNORECASE):

        return normalize_query_text(text)

    return short_action_query(text)


def build_query_variant(event, strategy):

    raw = normalize_query_text(event)

    cleaned = clean_vlm_event(raw)

    builders = {
        "raw_vlm": lambda: raw,
        "cleaned": lambda: cleaned,
        "short_action": lambda: short_action_query(cleaned),
        "the_video_shows": lambda: f"The video shows {lower_first(cleaned)}",
        "find_moment": lambda: f"Find the moment showing {lower_first(cleaned)}",
        "screen_text": lambda: screen_text_query(cleaned),
    }

    if strategy not in builders:

        raise ValueError(f"Unknown query variant strategy: {strategy}")

    return normalize_query_text(builders[strategy]())


def make_query_variants(event, strategy="raw_vlm"):

    if strategy == "all_unique":

        variants = [build_query_variant(event, name) for name in QUERY_VARIANT_DESCRIPTIONS]

    else:

        variants = [build_query_variant(event, strategy)]

    return list(dict.fromkeys([variant for variant in variants if variant]))


def preview_query_variants(query_table, strategies=QUERY_VARIANT_STRATEGIES_TO_RUN, n=12):

    rows = []

    for query_row in query_table.head(n).itertuples(index=False):

        for strategy in strategies:

            for variant in make_query_variants(query_row.event, strategy):

                rows.append(
                    {
                        "video_id": query_row.video_id,
                        "row_index": query_row.row_index,
                        "gt_time": query_row.gt_time,
                        "query_variant_strategy": strategy,
                        "query_variant": variant,
                    }
                )

    return pd.DataFrame(rows)


preview_query_variants(query_table, n=3)

,video_id,row_index,gt_time,query_variant_strategy,query_variant
0,6,1,00:00 - 00:10,raw_vlm,The video begins with a view of a wooden patio...
1,6,1,00:00 - 00:10,cleaned,"a view of a wooden patio area, featuring a tab..."
2,6,1,00:00 - 00:10,short_action,a view of a wooden patio area
3,6,1,00:00 - 00:10,the_video_shows,"The video shows a view of a wooden patio area,..."
4,6,1,00:00 - 00:10,find_moment,Find the moment showing a view of a wooden pat...
5,6,1,00:00 - 00:10,screen_text,a view of a wooden patio area
6,6,2,00:31 - 00:42,raw_vlm,"The camera follows the truck as it moves, capt..."
7,6,2,00:31 - 00:42,cleaned,"The camera follows the truck as it moves, capt..."
8,6,2,00:31 - 00:42,short_action,The camera follows the truck as it moves
9,6,2,00:31 - 00:42,the_video_shows,The video shows the camera follows the truck a...


In [77]:
def load_available_models(model_configs):

    models = {}

    skipped = []

    preferred_order = ["CG-DETR", "Moment-DETR"]

    ordered_names = preferred_order + [name for name in model_configs if name not in preferred_order]

    for name in ordered_names:

        cfg = model_configs[name]

        if not cfg["selected"]:

            continue

        checkpoint = cfg["checkpoint"]

        if not checkpoint.exists():

            skipped.append(
                {
                    "model": name,
                    "reason": f"Missing checkpoint: {checkpoint.relative_to(ROOT)}",
                }
            )
            continue

        slowfast_path = cfg.get("slowfast_path")

        if cfg["feature_name"] in {"clip_slowfast", "clip_slowfast_pann"}:

            if slowfast_path is None or not slowfast_path.exists():

                skipped.append(
                    {
                        "model": name,
                        "reason": "Missing SlowFast backbone: SLOWFAST_8x8_R50.pkl",
                    }
                )
                continue

        print(f"Loading {name} from {checkpoint.name} ...")

        models[name] = cfg["predictor_cls"](

                str(checkpoint),
                device=DEVICE,
                feature_name=cfg["feature_name"],
                slowfast_path=None if slowfast_path is None else str(slowfast_path),
                
            )
        
    return models, pd.DataFrame(skipped)


models, skipped_models = load_available_models(MODEL_CONFIGS)
print("Loaded:", list(models))
skipped_models

Loading CG-DETR from clip_cg_detr_qvhighlight.ckpt ...
Loading Moment-DETR from moment_detr_qvhighlight_clip.ckpt ...
Loading QD-DETR from qd_detr_qvhighlight_videoonly.ckpt ...
Loaded: ['CG-DETR', 'Moment-DETR', 'QD-DETR']


""


In [78]:
RETRIEVAL_RESULT_COLUMNS = [
    "model",
    "event",
    "query_variant_strategy",
    "query_variant",
    "gt_time",
    "subjectivity",
    "video_id",
    "segment_path",
    "segment_offset",
    "rank_in_segment",
    "pred_time",
    "pred_start",
    "pred_end",
    "score",
    "global_rank",
    "iou",
]


def _prediction_windows(prediction):
    if prediction is None:
        return None

    if isinstance(prediction, dict):
        return prediction.get("pred_relevant_windows")

    windows = getattr(prediction, "pred_relevant_windows", None)
    if windows is not None:
        return windows

    return prediction


def retrieve_for_model(model_name, model, query_table, query_variant_strategy="raw_vlm", encoded_cache=None):

    rows = []
    skipped_predictions = []

    # Cache encoded video features so each video/chunk is encoded only once per model.
    if encoded_cache is None:
        encoded_cache = {}

    # Process queries video by video.
    for video_id, video_queries in query_table.groupby("video_id"):

        # Get either the full video or its chunks, plus the full video duration.
        segments, video_duration = get_video_segments(video_id)

        print(f"{model_name} | video {video_id}: {len(video_queries)} queries, {len(segments)} segment(s)")

        # Loop over each full video segment or video chunk.
        for segment in segments:

            segment_path = segment["path"]

            # Create a unique cache key for this video/chunk.
            cache_key = str(segment_path.resolve())

            # If this segment was not encoded yet, encode it now.
            if cache_key not in encoded_cache:

                # Print which segment is being encoded.
                print(f"  encoding {segment_path.relative_to(ROOT)}")

                # Extract video features using the current model's encoder.
                encoded_cache[cache_key] = model.encode_video(str(segment_path))

            # Reuse the encoded video features.
            encoded_video = encoded_cache[cache_key]

            # Loop over each event/query for this video.
            for query_row in video_queries.itertuples(index=False):

                # Store predictions for this event/query.
                all_windows = []

                # Use one or more phrasings of the event query.
                for variant in make_query_variants(query_row.event, query_variant_strategy):

                    prediction = model.predict(variant, encoded_video)
                    windows = _prediction_windows(prediction)

                    if windows is None:
                        skipped_predictions.append(
                            {
                                "model": model_name,
                                "video_id": video_id,
                                "segment_path": str(segment_path.relative_to(ROOT)),
                                "event": query_row.event,
                                "query_variant_strategy": query_variant_strategy,
                                "query_variant": variant,
                                "reason": "predict returned no windows",
                            }
                        )
                        continue

                    # Loop over the model's predicted windows.
                    for rank, (start, end, score) in enumerate(windows, 1):

                        # Convert local segment start time into full-video start time.
                        pred_start = max(0.0, min(video_duration, float(start) + segment["offset"]))

                        # Convert local segment end time into full-video end time.
                        pred_end = max(0.0, min(video_duration, float(end) + segment["offset"]))

                        # Skip invalid predictions where the end comes before the start.
                        if pred_end <= pred_start:
                            continue

                        all_windows.append(
                            {
                              
                                "model": model_name,
                                "video_id": video_id,
                                "source_file": query_row.source_file,
                                "row_index": query_row.row_index,
                                "event": query_row.event,
                                "query_variant_strategy": query_variant_strategy,
                                "query_variant": variant,
                                "gt_start": float(query_row.gt_start),
                                "gt_end": float(query_row.gt_end),
                                "gt_time": query_row.gt_time,
                                "subjectivity": int(query_row.subjectivity),
                                "segment_path": str(segment_path.relative_to(ROOT)),
                                "segment_offset": segment["offset"],
                                "rank_in_segment": rank,
                                "pred_start": pred_start,
                                "pred_end": pred_end,
                                "pred_time": f"{seconds_to_mmss(pred_start)} - {seconds_to_mmss(pred_end)}",
                                "score": float(score),
                            }
                        )

                # Add this query's predictions to the full result list.
                rows.extend(all_windows)

    if skipped_predictions:
        skipped_df = pd.DataFrame(skipped_predictions)
        safe_model = model_name.lower().replace('-', '_')
        safe_strategy = query_variant_strategy.lower().replace('-', '_')
        skipped_path = OUTPUT_DIR / f"step4_skipped_predictions_{safe_model}_{safe_strategy}.csv"
        skipped_df.to_csv(skipped_path, index=False)
        print(f"{model_name}: skipped {len(skipped_df)} failed prediction calls; saved details to {skipped_path.relative_to(ROOT)}")

    raw = pd.DataFrame(rows)

    if raw.empty:
        return pd.DataFrame(columns=RETRIEVAL_RESULT_COLUMNS)

    final_rows = []

    group_cols = ["model", "query_variant_strategy", "video_id", "source_file", "row_index", "event"]

    # Group all predictions belonging to the same query.
    for _, group in raw.groupby(group_cols, sort=False):

        # Sort predictions by confidence score, highest first.
        windows = group.sort_values("score", ascending=False).to_dict("records")

        # Remove highly overlapping duplicate windows and keep top K.
        kept = nms_windows(windows, iou_threshold=NMS_IOU_THRESHOLD, max_keep=TOP_K_PER_QUERY)

        # Give the remaining predictions a global rank.
        for global_rank, item in enumerate(kept, 1):

            # Store rank after NMS.
            item["global_rank"] = global_rank

            # Compute overlap between prediction and ground truth.
            item["iou"] = temporal_iou(
                item["pred_start"],
                item["pred_end"],
                item["gt_start"],
                item["gt_end"],
            )

            final_rows.append(item)

    # Return final predictions as a dataframe.
    return pd.DataFrame(final_rows, columns=RETRIEVAL_RESULT_COLUMNS)


## 4. Run Retrieval

This is the main Step 4 execution cell. On CPU, the first run is slower because video features are encoded. Subsequent query predictions reuse cached features inside the cell.

In [79]:
all_results = []

for model_name, model in models.items():

    encoded_cache = {}

    for query_variant_strategy in QUERY_VARIANT_STRATEGIES_TO_RUN:

        print(f"Running {model_name} with query strategy: {query_variant_strategy}")

        model_results = retrieve_for_model(
            model_name,
            model,
            query_table,
            query_variant_strategy=query_variant_strategy,
            encoded_cache=encoded_cache,
        )

        if not model_results.empty:

            all_results.append(model_results)

if all_results:
    results_df = pd.concat(all_results, ignore_index=True)
    retrieval_predictions_path = OUTPUT_DIR / "step4_retrieval_predictions.csv"
    results_df.to_csv(retrieval_predictions_path, index=False)
    print(f"Saved retrieval predictions to {retrieval_predictions_path.relative_to(ROOT)}")
else:
    results_df = pd.DataFrame()
    print("No retrieval predictions to save.")

print(f"Prediction rows: {len(results_df)}")
results_df.head()

Running CG-DETR with query strategy: raw_vlm


Using existing chunks from video_6_3_chunks
CG-DETR | video 6: 7 queries, 3 segment(s)
  encoding video_6_3_chunks\chunk_0000.00_0107.57.mp4


FileNotFoundError: [WinError 2] The system cannot find the file specified

In [43]:
results_df[(results_df['model'] == 'Moment-DETR') & (results_df['query_variant_strategy'] == 'screen_text')][['event','query_variant_strategy','query_variant','pred_time','score','iou']]

KeyError: 'model'

In [14]:
if "query_variant_strategy" not in results_df.columns:
    results_df["query_variant_strategy"] = "raw_vlm"

ranked_results = results_df.sort_values(
    ["model", "query_variant_strategy", "video_id", "global_rank"]
)

best_by_query = (
    ranked_results.groupby(["model", "query_variant_strategy", "video_id", "event"], as_index=False)
    .agg(
        gt_time=("gt_time", "first"),
        subjectivity=("subjectivity", "first"),
        top1_query_variant=("query_variant", "first"),
        top1_time=("pred_time", "first"),
        top1_score=("score", "first"),
        top1_iou=("iou", "first"),
        best_iou_at_k=("iou", "max"),
    )
)

summary_df = (
    best_by_query.groupby(["model", "query_variant_strategy", "video_id"], as_index=False)
    .agg(
        queries=("event", "count"),
        mean_top1_iou=("top1_iou", "mean"),
        mean_best_iou_at_k=("best_iou_at_k", "mean"),
        recall_top1_iou_030=("top1_iou", lambda values: (values >= 0.30).mean()),
        recall_at_k_iou_030=("best_iou_at_k", lambda values: (values >= 0.30).mean()),
        recall_at_k_iou_050=("best_iou_at_k", lambda values: (values >= 0.50).mean()),
    )
    .sort_values(["model", "video_id", "mean_top1_iou", "mean_best_iou_at_k"], ascending=[True, True, False, False])
)

best_strategy_df = (
    summary_df.sort_values(["model", "video_id", "mean_best_iou_at_k"], ascending=[True, True,False])
    .groupby(["model", "video_id"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

display(best_strategy_df)
display(summary_df)
display(best_by_query.sort_values(["model", "query_variant_strategy", "video_id", "best_iou_at_k"], ascending=[True, True, True, False]).head(12))

best_by_query_path = OUTPUT_DIR / "step4_best_by_query.csv"
summary_metrics_path = OUTPUT_DIR / "step4_summary_metrics.csv"
best_strategy_path = OUTPUT_DIR / "step4_best_query_strategy.csv"
best_by_query.to_csv(best_by_query_path, index=False)
summary_df.to_csv(summary_metrics_path, index=False)
best_strategy_df.to_csv(best_strategy_path, index=False)
print(f"Saved per-query results to {best_by_query_path.relative_to(ROOT)}")
print(f"Saved summary metrics to {summary_metrics_path.relative_to(ROOT)}")
print(f"Saved best query strategy summary to {best_strategy_path.relative_to(ROOT)}")

,model,query_variant_strategy,video_id,queries,mean_top1_iou,mean_best_iou_at_k,recall_top1_iou_030,recall_at_k_iou_030,recall_at_k_iou_050
0,CG-DETR,screen_text,5,6,0.035909,0.287648,0.000000,0.500000,0.166667
1,Moment-DETR,screen_text,5,6,0.000000,0.311894,0.000000,0.500000,0.333333
2,QD-DETR,raw_vlm,5,6,0.135193,0.229721,0.166667,0.333333,0.166667


,model,query_variant_strategy,video_id,queries,mean_top1_iou,mean_best_iou_at_k,recall_top1_iou_030,recall_at_k_iou_030,recall_at_k_iou_050
5,CG-DETR,the_video_shows,5,6,0.138994,0.277007,0.166667,0.333333,0.333333
4,CG-DETR,short_action,5,6,0.070291,0.161421,0.000000,0.166667,0.000000
0,CG-DETR,cleaned,5,6,0.068750,0.268482,0.000000,0.333333,0.166667
2,CG-DETR,raw_vlm,5,6,0.068476,0.275469,0.000000,0.333333,0.166667
3,CG-DETR,screen_text,5,6,0.035909,0.287648,0.000000,0.500000,0.166667
1,CG-DETR,find_moment,5,6,0.034160,0.183817,0.000000,0.166667,0.166667
7,Moment-DETR,find_moment,5,6,0.121863,0.165357,0.000000,0.166667,0.000000
6,Moment-DETR,cleaned,5,6,0.084172,0.238394,0.000000,0.166667,0.166667
11,Moment-DETR,the_video_shows,5,6,0.083971,0.136661,0.000000,0.000000,0.000000
8,Moment-DETR,raw_vlm,5,6,0.083004,0.135968,0.000000,0.000000,0.000000


,model,query_variant_strategy,video_id,event,gt_time,subjectivity,top1_query_variant,top1_time,top1_score,top1_iou,best_iou_at_k
1,CG-DETR,cleaned,5,"The man, identified as Dave Campbell, provides...",00:06 - 00:18,1,"The man, identified as Dave Campbell, provides...",00:29 - 01:02,0.9699,0.000000,0.658956
3,CG-DETR,cleaned,5,The video concludes with a title screen displa...,01:47 - 01:50,1,"a title screen displaying ""GMC Certified Servi...",00:30 - 00:46,0.9905,0.000000,0.466505
2,CG-DETR,cleaned,5,The video begins with a title screen displayin...,00:00 - 00:04,1,"a title screen displaying ""GMC Certified Servi...",00:00 - 00:19,0.9870,0.206289,0.206289
5,CG-DETR,cleaned,5,The video emphasizes the importance of rotatin...,00:38 - 00:43,1,The video emphasizes the importance of rotatin...,00:34 - 00:58,0.9916,0.206210,0.206210
0,CG-DETR,cleaned,5,The final segment shows a man in a white shirt...,01:32 - 01:37,1,a man in a white shirt standing in front of a ...,00:05 - 00:19,0.9925,0.000000,0.072931
4,CG-DETR,cleaned,5,The video continues with a close-up of a tire ...,00:21 - 00:37,1,"a close-up of a tire being rotated, followed b...",00:46 - 00:59,0.9956,0.000000,0.000000
9,CG-DETR,find_moment,5,The video concludes with a title screen displa...,01:47 - 01:50,1,Find the moment showing a title screen display...,00:31 - 00:47,0.9873,0.000000,0.513119
11,CG-DETR,find_moment,5,The video emphasizes the importance of rotatin...,00:38 - 00:43,1,Find the moment showing the video emphasizes t...,01:08 - 01:17,0.9853,0.000000,0.208638
8,CG-DETR,find_moment,5,The video begins with a title screen displayin...,00:00 - 00:04,1,Find the moment showing a title screen display...,00:00 - 00:19,0.9781,0.204962,0.204962
7,CG-DETR,find_moment,5,"The man, identified as Dave Campbell, provides...",00:06 - 00:18,1,"Find the moment showing the man, identified as...",00:34 - 00:59,0.9975,0.000000,0.107850


Saved per-query results to outputs\step4_best_by_query.csv
Saved summary metrics to outputs\step4_summary_metrics.csv
Saved best query strategy summary to outputs\step4_best_query_strategy.csv


In [30]:
best_strategy_df = (
    summary_df.sort_values(["model", "video_id", "mean_top1_iou", "mean_best_iou_at_k"], ascending=[True, True, False, False])
    .groupby(["model", "video_id"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)


best_strategy_df[['model','query_variant_strategy', 'mean_top1_iou','mean_best_iou_at_k','recall_top1_iou_030','recall_at_k_iou_030']]

,model,query_variant_strategy,mean_top1_iou,mean_best_iou_at_k,recall_top1_iou_030,recall_at_k_iou_030
0,CG-DETR,the_video_shows,0.138994,0.277007,0.166667,0.333333
1,Moment-DETR,find_moment,0.121863,0.165357,0.000000,0.166667
2,QD-DETR,the_video_shows,0.137589,0.202521,0.166667,0.333333


## 5. Inspect Predictions and Failure Cases

In [16]:
# #Show the top-1 prediction for each query, sorted by model/video and confidence score.
# top1_rows = results_df[results_df["global_rank"] == 1].copy()
# top1_rows = top1_rows.sort_values(["model", "query_variant_strategy", "video_id", "iou", "score"])

# #IoU sorted ascending = worst predictions first


# top1_rows[
#     [
#         "model",
#         "query_variant_strategy",
#         "video_id",
#         "event",
#         "gt_time",
#         "pred_time",
#         "score",
#         "iou",
#         "query_variant",
#     ]
# ]

The biggest pattern: top-1 is weak for all models. Most mean_top1_iou values are low, and many recall_top1_iou_030 values are 0. That means the model often does not put the correct moment first.

ut with K = 10, performance improves a lot. For example:

Moment-DETR + short_action:
recall_at_k_iou_030 = 0.833333
That means 5 out of 6 queries had a decent match somewhere in the top 10 predictions. So the model is often finding the right region, just ranking it too low.

Comparing K = 5 vs K = 10:

Moment-DETR short_action
K=5:  mean_best_iou_at_k = 0.279511
K=10: mean_best_iou_at_k = 0.464786
That is a big jump. It suggests the good Moment-DETR predictions are often ranked between positions 6 and 10.

For CG-DETR, screen_text becomes best at K = 10:

CG-DETR screen_text:
K=10 mean_best_iou_at_k = 0.400121
K=10 recall_at_k_iou_030 = 0.666667
This makes sense because your video has title screens and displayed text like "GMC Certified Service". The screen_text variant helps the model focus on visual text/title moments.

For Moment-DETR, short_action is strongest at K = 10:

Moment-DETR short_action:
mean_best_iou_at_k = 0.464786
recall_at_k_iou_030 = 0.833333
This suggests Moment-DETR benefits from simpler, cleaner action descriptions instead of long VLM-style sentences.

QD-DETR looks weakest overall. Its best K = 10 result is:

QD-DETR cleaned:
mean_best_iou_at_k = 0.268700
recall_at_k_iou_030 = 0.500000
So it finds some matches, but not as strongly as Moment-DETR or CG-DETR here.

One important insight: find_moment is not consistently helpful. Even though it sounds like a retrieval instruction, the models were trained more like text-video matching systems, so plain descriptions such as short_action, cleaned, or the_video_shows can work better than instruction wording.

My read:

Best overall with K=10: Moment-DETR + short_action
Best CG-DETR variant: screen_text
Best QD-DETR variant: cleaned
Most stable practical setting: K=10, then evaluate best_iou_at_k
For your report, I would say: the models often retrieve relevant windows but rank them poorly, so increasing K from 5 to 10 reveals much better recall. Query phrasing matters: concise action-based queries help Moment-DETR, while text-focused queries help CG-DETR on title-screen events.


## 6. Video Summary

This final step turns the retrieved moments into a compact video-level summary. It uses the best available retrieval model, keeps one high-scoring window per salient event, removes overlapping duplicate windows, orders the selected moments chronologically, and exports both a readable text summary and optional concatenated summary clips.

In [55]:
# cleaned, short_action, the_video_shows, find_moment, screen_text, raw_vlm

SUMMARY_PREFERRED_MODEL = "CG-DETR" #Moment-DETR
SUMMARY_PREFERRED_QUERY_STRATEGY = "the_video_shows" 
SUMMARY_MOMENTS_PER_VIDEO = 6
SUMMARY_PADDING_SECONDS = 1.0
SUMMARY_OVERLAP_THRESHOLD = 0.1


def pick_summary_model(results, preferred_model=SUMMARY_PREFERRED_MODEL):
    available = sorted(results["model"].dropna().unique())

    if preferred_model in available:
        return preferred_model

    if "iou" in results.columns and not results["iou"].dropna().empty:
        return results.groupby("model")["iou"].mean().sort_values(ascending=False).index[0]

    return available[0] if available else None


def pick_summary_query_strategy(results, model_name, video_id=None, preferred_strategy=SUMMARY_PREFERRED_QUERY_STRATEGY):
    if "query_variant_strategy" not in results.columns:
        return None

    model_results = results[results["model"] == model_name]

    if video_id is not None and "video_id" in model_results.columns:
        model_results = model_results[model_results["video_id"] == video_id]

    available = sorted(model_results["query_variant_strategy"].dropna().unique())

    if preferred_strategy in available:
        return preferred_strategy

    if "best_strategy_df" in globals() and not best_strategy_df.empty:
        best_rows = best_strategy_df[best_strategy_df["model"] == model_name]

        if video_id is not None:
            if "video_id" in best_rows.columns:
                best_rows = best_rows[best_rows["video_id"] == video_id]
            else:
                best_rows = pd.DataFrame()

        if not best_rows.empty:
            return best_rows.sort_values("mean_best_iou_at_k", ascending=False)["query_variant_strategy"].iloc[0]

    if "iou" in model_results.columns and not model_results["iou"].dropna().empty:
        return model_results.groupby("query_variant_strategy")["iou"].mean().sort_values(ascending=False).index[0]

    return available[0] if available else None


def select_summary_moments(
    results,
    model_name=None,
    query_variant_strategy=None,
    moments_per_video=SUMMARY_MOMENTS_PER_VIDEO,
    padding=SUMMARY_PADDING_SECONDS,
    overlap_threshold=SUMMARY_OVERLAP_THRESHOLD,
):
    if results.empty:
        return pd.DataFrame()

    model_name = model_name or pick_summary_model(results)

    if model_name is None:
        return pd.DataFrame()

    required_cols = {"video_id", "global_rank", "pred_start", "pred_end", "score", "event"}
    missing_cols = sorted(required_cols - set(results.columns))

    if missing_cols:
        raise ValueError(
            "Step 6 expects the raw Step 4 retrieval results_df. "
            f"Missing required column(s): {missing_cols}. Re-run the Step 4 retrieval cell."
        )

    working_results = results.copy()

    if "subjectivity" not in working_results.columns:
        working_results["subjectivity"] = 1

    candidates = working_results[(working_results["model"] == model_name) & (working_results["global_rank"] <= 5)].copy()

    if query_variant_strategy is not None and "query_variant_strategy" in candidates.columns:
        candidates = candidates[candidates["query_variant_strategy"] == query_variant_strategy]


    if candidates.empty:
        return pd.DataFrame()

    candidates = candidates.sort_values(
        ["video_id", "subjectivity", "score"],
        ascending=[True, False, False],
    )

    summary_rows = []

    for video_id, group in candidates.groupby("video_id", sort=True):
        selected_strategy = query_variant_strategy or pick_summary_query_strategy(
            working_results,
            model_name,
            video_id=video_id,
        )

        if selected_strategy is not None and "query_variant_strategy" in group.columns:
            group = group[group["query_variant_strategy"] == selected_strategy]

        if group.empty:
            continue

        video_path = VIDEO_DIR / f"video_{int(video_id)}.mp4"
        video_duration = get_video_duration(video_path)
        selected = []
        selected_event_keys = set()

        for item in group.to_dict("records"):
            event_key = normalize_query_text(item["event"]).casefold()

            if event_key in selected_event_keys:
                continue

            summary_start = max(0.0, float(item["pred_start"]) - padding)
            summary_end = min(video_duration, float(item["pred_end"]) + padding)

            if summary_end <= summary_start:
                continue

            overlaps_existing_summary = any(
                temporal_iou(
                    summary_start,
                    summary_end,
                    previous["summary_start"],
                    previous["summary_end"],
                ) >= overlap_threshold
                for previous in selected
            )

            if overlaps_existing_summary:
                continue

            item["summary_start"] = summary_start
            item["summary_end"] = summary_end
            item["summary_time"] = f"{seconds_to_mmss(summary_start)} - {seconds_to_mmss(summary_end)}"
            item["summary_duration"] = summary_end - summary_start
            selected.append(item)
            selected_event_keys.add(event_key)

            if len(selected) >= moments_per_video:
                break

        for summary_order, item in enumerate(sorted(selected, key=lambda row: row["summary_start"]), 1):
            item["summary_order"] = summary_order
            summary_rows.append(item)

    summary = pd.DataFrame(summary_rows)

    if summary.empty:
        return summary

    return summary[
        [
            "model",
            "video_id",
            "query_variant_strategy",
            "summary_order",
            "summary_time",
            "event",
            "score",
            "subjectivity",
            "pred_time",
            "gt_time",
            "iou",
            "summary_start",
            "summary_end",
            "summary_duration",
        ]
    ]


if "results_df" not in globals() or results_df.empty:
    summary_plan_df = pd.DataFrame()
    print("Run Step 4 first; no retrieval results are available for Step 6.")
else:
    summary_model = pick_summary_model(results_df)
    summary_query_strategy = SUMMARY_PREFERRED_QUERY_STRATEGY
    summary_plan_df = select_summary_moments(
        results_df,
        model_name=summary_model,
        query_variant_strategy=summary_query_strategy,
    )
    strategy_label = summary_query_strategy or "best per video"
    print(f"Summary model: {summary_model} | query strategy: {strategy_label}")

    if summary_plan_df.empty:
        print("No summary moments selected.")
    else:
        display(
            summary_plan_df[
                [
                    "video_id",
                    "query_variant_strategy",
                    "summary_order",
                    "summary_time",
                    "event",
                    "score",
                    "subjectivity",
                    "iou",
                ]
            ]
        )

Summary model: CG-DETR | query strategy: the_video_shows


,video_id,query_variant_strategy,summary_order,summary_time,event,score,subjectivity,iou
0,5,the_video_shows,1,00:03 - 00:20,The final segment shows a man in a white shirt...,0.9923,1,0.0
1,5,the_video_shows,2,00:30 - 00:47,The video concludes with a title screen displa...,0.9915,1,0.0
2,5,the_video_shows,3,00:54 - 01:01,The video emphasizes the importance of rotatin...,0.4687,1,0.0
3,5,the_video_shows,4,01:07 - 01:18,The video continues with a close-up of a tire ...,0.9885,1,0.0
4,5,the_video_shows,5,01:24 - 01:33,The video begins with a title screen displayin...,0.9703,1,0.0


In [56]:
def format_video_summary(summary_plan, video_id):
    rows = summary_plan[summary_plan["video_id"] == video_id].sort_values("summary_order")

    if rows.empty:
        return f"Video {int(video_id)}: no summary moments selected."

    model_name = rows["model"].iloc[0]
    strategy = rows["query_variant_strategy"].iloc[0] if "query_variant_strategy" in rows.columns else "unknown"
    lines = [f"Video {int(video_id)} retrieval-based summary ({len(rows)} moments, model={model_name}, query_strategy={strategy}):"]

    for row in rows.itertuples(index=False):
        lines.append(f"{int(row.summary_order)}. {row.summary_time}: {row.event}")

    return "\n".join(lines)


summary_text_paths = []

if summary_plan_df.empty:
    print("No summary moments selected.")
else:
    for video_id in sorted(summary_plan_df["video_id"].unique()):
        summary_text = format_video_summary(summary_plan_df, video_id)
        summary_text_path = OUTPUT_DIR / f"video_{int(video_id)}_retrieval_summary.txt"
        summary_text_path.write_text(summary_text + "\n", encoding="utf-8")
        summary_text_paths.append(
            {
                "video_id": int(video_id),
                "summary_text_path": str(summary_text_path.relative_to(ROOT)),
            }
        )
        print(summary_text)
        print()

    display(pd.DataFrame(summary_text_paths))

Video 5 retrieval-based summary (5 moments, model=CG-DETR, query_strategy=the_video_shows):
1. 00:03 - 00:20: The final segment shows a man in a white shirt standing in front of a dark blue background with a GMC truck, with the text "GMC Certified Service" and "gmccertifiedservice.com" appearing on the screen
2. 00:30 - 00:47: The video concludes with a title screen displaying "GMC Certified Service" and "Replacing Your Tires
3. 00:54 - 01:01: The video emphasizes the importance of rotating tires every 7,500 miles
4. 01:07 - 01:18: The video continues with a close-up of a tire being rotated, followed by a comparison of good and bad tire treads
5. 01:24 - 01:33: The video begins with a title screen displaying "GMC Certified Service" and "Replacing Your Tires." It transitions to a man in a white shirt with the GMC logo standing in front of a dark blue background with a GMC truck



,video_id,summary_text_path
0,5,outputs\video_5_retrieval_summary.txt


In [58]:
BUILD_SUMMARY_VIDEOS = True
SUMMARY_INCLUDE_AUDIO = False
SUMMARY_VIDEO_CODEC = "libx264"


def _make_subclip(clip, start, end):
    return clip.subclipped(start, end) if hasattr(clip, "subclipped") else clip.subclip(start, end)


def make_summary_video(video_id, summary_plan, output_dir=OUTPUT_DIR, include_audio=SUMMARY_INCLUDE_AUDIO):
    rows = summary_plan[summary_plan["video_id"] == video_id].sort_values("summary_order")

    if rows.empty:
        return None

    try:
        from moviepy import concatenate_videoclips
    except ImportError:
        from moviepy.editor import concatenate_videoclips

    source_path = VIDEO_DIR / f"video_{int(video_id)}.mp4"
    source_clip = VideoFileClip(str(source_path))
    clips = []
    final_clip = None

    try:
        for row in rows.itertuples(index=False):
            clips.append(_make_subclip(source_clip, float(row.summary_start), float(row.summary_end)))

        final_clip = concatenate_videoclips(clips, method="compose")
        output_dir.mkdir(exist_ok=True)
        output_path = output_dir / f"video_{int(video_id)}_retrieval_summary.mp4"
        final_clip.write_videofile(
            str(output_path),
            codec=SUMMARY_VIDEO_CODEC,
            audio=include_audio,
            logger=None,
        )
        return output_path

    finally:
        if final_clip is not None:
            final_clip.close()

        for clip in clips:
            clip.close()

        source_clip.close()


summary_video_paths = []

if BUILD_SUMMARY_VIDEOS and not summary_plan_df.empty:
    for video_id in sorted(summary_plan_df["video_id"].unique()):
        summary_video_path = make_summary_video(video_id, summary_plan_df)

        if summary_video_path is not None:
            summary_video_paths.append(
                {
                    "video_id": int(video_id),
                    "summary_video_path": str(summary_video_path.relative_to(ROOT)),
                }
            )

if summary_video_paths:
    summary_video_df = pd.DataFrame(summary_video_paths)
    display(summary_video_df)

    for item in summary_video_paths:
        display(Video(str(ROOT / item["summary_video_path"]), embed=True))
else:
    print("No summary videos were rendered. Set BUILD_SUMMARY_VIDEOS = True and run Step 6 after Step 4.")

,video_id,summary_video_path
0,5,outputs\video_5_retrieval_summary.mp4
